In [4]:
# %%
"""
=========================================================================================
PROJECT: LANDSLIDE FORECASTING - Bi-LSTM ULTIMATE (Safety Tuned)
=========================================================================================
[MODEL] Bidirectional LSTM
[OPTIMIZATION] Tune Thresholds on Validation Set to Maximize F1-Critical > F1-Warning
"""

import os
import glob
import time
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings('ignore')

# ====================================================================
# 1. CONFIGURATION
# ====================================================================
class Config:
    DATA_DIR = "./data/"
    MODEL_DIR = "./model/bilstm_tuned/"
    SCALER_PATH = "./model/bilstm_tuned_scaler.save"
    
    #HORIZONS = [0, 30, 60, 180, 360, 720, 1440]
    HORIZONS = [0, 30]
    SEQUENCE_LENGTH = 60
    TEST_KEYWORDS = ['monitoring_data_2025-11-22_19-57-13_prepared']
    
    RAW_COLS = ['rain', 'soil', 'temp', 'humi', 'geo']
    LABEL_COL = 'label'
    LABEL_MAP = {'normal': 0, 'warning': 1, 'critical': 2}
    
    TARGET_SAMPLES = 5000 
    BATCH_SIZE = 64
    EPOCHS = 100
    LEARNING_RATE = 1e-4
    CLASS_WEIGHTS = {0: 1.0, 1: 2.0, 2: 3.0}

cfg = Config()
if not os.path.exists(cfg.MODEL_DIR): os.makedirs(cfg.MODEL_DIR)
np.random.seed(42); tf.random.set_seed(42); random.seed(42)

def log(msg): print(f"[INFO] {time.strftime('%H:%M:%S')} - {msg}")
print("✅ Config Loaded.")

# ====================================================================
# 2. DATA LOADING & PREP (Standard)
# ====================================================================
def load_data():
    log("Scanning Data...")
    files = glob.glob(os.path.join(cfg.DATA_DIR, "*.csv"))
    if not files: files = glob.glob(os.path.join(cfg.DATA_DIR, "**/*.csv"), recursive=True)
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f)
            df.columns = [str(c).lower().strip().replace('.1', '') for c in df.columns]
            rename_map = {'temperature':'temp', 'hum':'humi', 'humidity':'humi', 'devid':'devID', 'time':'timestamp', 'date':'timestamp'}
            new_cols = {c: rename_map[c] for c in df.columns if c in rename_map}
            if new_cols: df = df.rename(columns=new_cols)
            if 'devID' in df.columns: df['devID'] = df['devID'].astype(str).str.extract('(\d+)')[0].astype(float).fillna(0).astype(int)
            if 'timestamp' in df.columns: df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
            for c in cfg.RAW_COLS: 
                if c not in df.columns: df[c] = 0.0
            filename = os.path.basename(f)
            df['is_test'] = any(k in filename for k in cfg.TEST_KEYWORDS)
            dfs.append(df)
            print(f"   + Loaded: {filename} [{'TEST' if df['is_test'].iloc[0] else 'TRAIN'}]")
        except: pass
    return pd.concat(dfs, ignore_index=True) if dfs else None

def generate_features(df):
    log("Feature Engineering...")
    df = df.sort_values(['devID', 'timestamp']).reset_index(drop=True)
    df_list = []
    for dev, g in df.groupby('devID'):
        g = g.set_index('timestamp')
        g = g[~g.index.duplicated(keep='first')]
        if len(g) > 0: g = g.resample('1T').asfreq()
        g[cfg.RAW_COLS] = g[cfg.RAW_COLS].interpolate(limit_direction='both').fillna(0)
        g['rain_1h'] = g['rain'].rolling(60, min_periods=1).sum()
        g['soil_rate'] = g['soil'].diff().fillna(0)
        g['soil_acc']  = g['soil'].diff().diff().fillna(0)
        g['rain_6h'] = g['rain'].rolling(360, min_periods=1).sum()
        g['soil_trend_6h'] = g['soil'] - g['soil'].shift(360).fillna(method='bfill')
        g['rain_24h'] = g['rain'].rolling(1440, min_periods=1).sum()
        g['soil_max_24h'] = g['soil'].rolling(1440, min_periods=1).max()
        g['rain_x_soil'] = g['rain'] * g['soil']
        g['rain_x_geo']  = g['rain'] * g['geo'].abs()
        if cfg.LABEL_COL in g.columns:
            g[cfg.LABEL_COL] = g[cfg.LABEL_COL].fillna('normal').astype(str).str.lower().str.strip()
            g[cfg.LABEL_COL] = g[cfg.LABEL_COL].map(cfg.LABEL_MAP).fillna(0).astype(int)
        else: g[cfg.LABEL_COL] = 0
        g['is_test'] = g['is_test'].fillna(method='ffill').fillna(method='bfill')
        g['devID'] = dev
        df_list.append(g.reset_index())
    return pd.concat(df_list, ignore_index=True).fillna(0)

df_train_raw = load_data()
df_proc = generate_features(df_train_raw)

def create_sequences(df, feature_cols, horizon):
    Xs, ys, is_tests = [], [], []
    for dev, g in df.groupby('devID'):
        data = g[feature_cols].values
        labels = g[cfg.LABEL_COL].shift(-horizon).values
        test_flag = g['is_test'].values
        valid_len = len(g) - horizon
        if valid_len < cfg.SEQUENCE_LENGTH: continue
        for i in range(valid_len - cfg.SEQUENCE_LENGTH):
            Xs.append(data[i : i+cfg.SEQUENCE_LENGTH])
            ys.append(labels[i+cfg.SEQUENCE_LENGTH])
            is_tests.append(test_flag[i+cfg.SEQUENCE_LENGTH])
    return np.array(Xs), np.array(ys), np.array(is_tests)

def balance_data(X, y):
    print(f"   Original counts: {Counter(y)}")
    X_bal, y_bal = [], []
    for cls in np.unique(y):
        if np.isnan(cls): continue
        indices = np.where(y == cls)[0]
        count = len(indices)
        replace = count < cfg.TARGET_SAMPLES
        chosen_indices = np.random.choice(indices, cfg.TARGET_SAMPLES, replace=replace)
        X_bal.append(X[chosen_indices])
        y_bal.append(y[chosen_indices])
    X_bal = np.concatenate(X_bal)
    y_bal = np.concatenate(y_bal)
    perm = np.random.permutation(len(X_bal))
    return X_bal[perm], y_bal[perm]

def build_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    
    # Bi-LSTM 1
    x = Bidirectional(LSTM(128, return_sequences=True))(inputs)
    x = BatchNormalization()(x)  # <--- แก้ตรงนี้ (เพิ่ม (x))
    x = Dropout(0.3)(x)
    
    # Bi-LSTM 2
    x = Bidirectional(LSTM(64, return_sequences=False))(x)
    x = BatchNormalization()(x)  # <--- แก้ตรงนี้ (เพิ่ม (x))
    x = Dropout(0.3)(x)
    
    # Dense Layer
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    
    # Output
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=cfg.LEARNING_RATE), 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

# ====================================================================
# 6. THRESHOLD TUNING (NEW!)
# ====================================================================
def optimize_thresholds_for_safety(model, X_val, y_val_true):
    """
    หาจุดตัด (Threshold) ที่ทำให้ F1-Critical สูงสุด
    รองลงมาคือ F1-Warning
    """
    log("🔧 Tuning Safety Thresholds on Validation Set...")
    probs = model.predict(X_val, verbose=0)
    
    best_f1_crit = -1
    best_f1_warn = -1
    best_thresh_crit = 0.5
    best_thresh_warn = 0.5
    
    # Grid Search for Thresholds
    # ลองจุดตัด Critical ตั้งแต่ 0.1 ถึง 0.9
    for t_crit in np.arange(0.1, 0.9, 0.05):
        # ลองจุดตัด Warning ตั้งแต่ 0.1 ถึง 0.9
        for t_warn in np.arange(0.1, 0.9, 0.05):
            
            # Apply Logic
            preds = np.zeros(len(probs), dtype=int)
            
            # 1. Priority Critical
            crit_mask = probs[:, 2] >= t_crit
            preds[crit_mask] = 2
            
            # 2. Priority Warning (ถ้าไม่ Crit)
            # ต้องมี prob warning > t_warn และไม่ได้เป็น crit
            warn_mask = (~crit_mask) & (probs[:, 1] >= t_warn)
            preds[warn_mask] = 1
            
            # 3. Normal (ที่เหลือ)
            
            # Calc Metrics
            f1_crit = f1_score(y_val_true, preds, labels=[2], average=None)[0] # เฉพาะ class 2
            f1_warn = f1_score(y_val_true, preds, labels=[1], average=None)[0] # เฉพาะ class 1
            
            # Optimization Goal: Max Crit, Then Warn
            # ถ้า Crit ดีขึ้น -> เก็บเลย
            if f1_crit > best_f1_crit:
                best_f1_crit = f1_crit
                best_f1_warn = f1_warn
                best_thresh_crit = t_crit
                best_thresh_warn = t_warn
            # ถ้า Crit เท่าเดิม แต่ Warn ดีขึ้น -> เก็บ
            elif f1_crit == best_f1_crit and f1_warn > best_f1_warn:
                best_f1_warn = f1_warn
                best_thresh_crit = t_crit
                best_thresh_warn = t_warn
                
    print(f"   🔥 Best Thresholds -> Crit: {best_thresh_crit:.2f}, Warn: {best_thresh_warn:.2f}")
    print(f"      Val Scores -> F1-Crit: {best_f1_crit:.4f}, F1-Warn: {best_f1_warn:.4f}")
    
    return best_thresh_crit, best_thresh_warn

def predict_with_tuned_thresholds(model, X, t_crit, t_warn):
    probs = model.predict(X, verbose=0)
    preds = np.zeros(len(probs), dtype=int)
    
    # Apply tuned thresholds
    crit_mask = probs[:, 2] >= t_crit
    preds[crit_mask] = 2
    
    warn_mask = (~crit_mask) & (probs[:, 1] >= t_warn)
    preds[warn_mask] = 1
    
    return preds

# ====================================================================
# 7. MAIN LOOP
# ====================================================================
def train_system():
    FINAL_FEATURES = cfg.RAW_COLS + ['rain_1h', 'soil_rate', 'soil_acc', 'rain_6h', 'soil_trend_6h', 'rain_24h', 'soil_max_24h', 'rain_x_soil', 'rain_x_geo']
    
    log("Scaling Data...")
    scaler = RobustScaler()
    train_mask = ~df_proc['is_test']
    scaler.fit(df_proc.loc[train_mask, FINAL_FEATURES])
    df_proc[FINAL_FEATURES] = scaler.transform(df_proc[FINAL_FEATURES])
    joblib.dump(scaler, cfg.SCALER_PATH)
    
    results_log = []

    for h in cfg.HORIZONS:
        print(f"\n{'='*60}")
        print(f"🚀 HORIZON: +{h/60:.1f} Hours ({h} mins)")
        print(f"{'='*60}")
        
        # 1. Prepare
        X_all, y_all, test_mask = create_sequences(df_proc, FINAL_FEATURES, h)
        X_train = X_all[~test_mask]; y_train = y_all[~test_mask]
        X_test  = X_all[test_mask];  y_test  = y_all[test_mask]
        
        valid_tr = ~np.isnan(y_train); X_train = X_train[valid_tr]; y_train = y_train[valid_tr]
        valid_ts = ~np.isnan(y_test);  X_test = X_test[valid_ts];   y_test = y_test[valid_ts]
        
        # Val Split (Random is OK for internal validation)
        X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
        
        # 2. Balance Train
        print("⚖️ Balancing Train Data...")
        X_train_bal, y_train_bal = balance_data(X_train, y_train)
        
        # Encode
        y_train_cat = to_categorical(y_train_bal, 3)
        y_val_cat = to_categorical(y_val, 3)
        
        # Weights
        safety_factor = 1.0 + (h/1440.0)
        cw = {k: v * safety_factor for k, v in cfg.CLASS_WEIGHTS.items()}
        
        # 3. Train
        model = build_model((cfg.SEQUENCE_LENGTH, len(FINAL_FEATURES)), 3)
        save_path = os.path.join(cfg.MODEL_DIR, f"bilstm_h{h}.h5")
        
        cbs = [
            EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
            ReduceLROnPlateau(patience=5, factor=0.5, monitor='val_loss'),
            ModelCheckpoint(save_path, save_best_only=True, monitor='val_accuracy', verbose=0)
        ]
        
        model.fit(X_train_bal, y_train_cat, validation_data=(X_val, y_val_cat),
                  epochs=cfg.EPOCHS, batch_size=cfg.BATCH_SIZE, 
                  class_weight=cw, callbacks=cbs, verbose=1)
        
        # 4. TUNE THRESHOLDS (Validation Set)
        # แปลง y_val_cat กลับเป็น index (0,1,2) เพื่อเทียบกับ prediction
        y_val_indices = np.argmax(y_val_cat, axis=1)
        t_crit, t_warn = optimize_thresholds_for_safety(model, X_val, y_val_indices)
        
        # Save Thresholds (Save เป็น Text หรือ JSON คู่กับโมเดล)
        with open(os.path.join(cfg.MODEL_DIR, f"thresh_h{h}.txt"), "w") as f:
            f.write(f"{t_crit},{t_warn}")
        
        # 5. FINAL EVALUATION (Test Set using Tuned Thresholds)
        log(f"Evaluating Test Set with Tuned Thresholds...")
        y_pred = predict_with_tuned_thresholds(model, X_test, t_crit, t_warn)
        
        report = classification_report(y_test, y_pred, target_names=['Normal', 'Warning', 'Critical'], output_dict=True, zero_division=0)
        f1_norm = report['Normal']['f1-score']
        f1_warn = report['Warning']['f1-score']
        f1_crit = report['Critical']['f1-score']
        acc = accuracy_score(y_test, y_pred)
        
        print(f"   📊 Accuracy : {acc:.4f}")
        print(f"   🟢 F1-Norm  : {f1_norm:.4f}")
        print(f"   🟡 F1-Warn  : {f1_warn:.4f}")
        print(f"   🔴 F1-Crit  : {f1_crit:.4f}")
        
        results_log.append({'Horizon': h, 'Acc': acc, 'F1_Norm': f1_norm, 'F1_Warn': f1_warn, 'F1_Crit': f1_crit})

    summary = pd.DataFrame(results_log)
    summary.to_csv("./model/bilstm_tuned_performance.csv", index=False)
    print("\n=== FINAL SUMMARY ===")
    print(summary)
    
    plt.figure(figsize=(10, 6))
    plt.plot(summary['Horizon'], summary['F1_Crit'], marker='o', color='red', label='Critical F1')
    plt.plot(summary['Horizon'], summary['F1_Warn'], marker='s', color='orange', label='Warning F1')
    plt.title("Bi-LSTM Forecast Performance (Safety Tuned)")
    plt.xlabel("Minutes Ahead")
    plt.ylabel("F1 Score")
    plt.legend(); plt.grid(True)
    plt.show()

if __name__ == "__main__":
    train_system()

✅ Config Loaded.
[INFO] 08:24:22 - Scanning Data...
   + Loaded: final_moving_avg_devID109_test_prepared.csv [TRAIN]
   + Loaded: label_for_dev108_test_prepared.csv [TRAIN]
   + Loaded: label_for_dev109_test_prepared.csv [TRAIN]
   + Loaded: monitoring_data_2025-11-22_19-57-13_prepared.csv [TEST]
   + Loaded: moving_avg_for_train_prepared.csv [TRAIN]
[INFO] 08:24:22 - Feature Engineering...
[INFO] 08:24:22 - Scaling Data...

🚀 HORIZON: +0.0 Hours (0 mins)
⚖️ Balancing Train Data...
   Original counts: Counter({np.int64(0): 27795, np.int64(1): 776, np.int64(2): 528})
Epoch 1/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.5982 - loss: 1.8850

235/235 ━━━━━━━━━━━━━━━━━━━━ 24s 86ms/step - accuracy: 0.5990 - loss: 1.8810 - val_accuracy: 0.8671 - val_loss: 0.3873 - learning_rate: 1.0000e-04
Epoch 2/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.7901 - loss: 1.0320

235/235 ━━━━━━━━━━━━━━━━━━━━ 21s 90ms/step - accuracy: 0.7902 - loss: 1.0315 - val_accuracy: 0.8913 - val_loss: 0.3828 - learning_rate: 1.0000e-04
Epoch 3/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.8161 - loss: 0.9012

235/235 ━━━━━━━━━━━━━━━━━━━━ 22s 92ms/step - accuracy: 0.8162 - loss: 0.9009 - val_accuracy: 0.8948 - val_loss: 0.3793 - learning_rate: 1.0000e-04
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 21s 87ms/step - accuracy: 0.8258 - loss: 0.8269 - val_accuracy: 0.8932 - val_loss: 0.3760 - learning_rate: 1.0000e-04
Epoch 5/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.8442 - loss: 0.7628

235/235 ━━━━━━━━━━━━━━━━━━━━ 21s 88ms/step - accuracy: 0.8442 - loss: 0.7627 - val_accuracy: 0.8966 - val_loss: 0.3600 - learning_rate: 1.0000e-04
Epoch 6/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.8553 - loss: 0.7274

...

235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 85ms/step - accuracy: 0.9142 - loss: 0.4100 - val_accuracy: 0.8894 - val_loss: 0.3930 - learning_rate: 1.0000e-04
Epoch 28/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.9183 - loss: 0.4032

235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 84ms/step - accuracy: 0.9183 - loss: 0.4032 - val_accuracy: 0.8997 - val_loss: 0.3808 - learning_rate: 1.0000e-04
Epoch 29/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 83ms/step - accuracy: 0.9232 - loss: 0.3760 - val_accuracy: 0.8986 - val_loss: 0.3849 - learning_rate: 1.0000e-04
Epoch 30/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 19s 83ms/step - accuracy: 0.9234 - loss: 0.3827 - val_accuracy: 0.8952 - val_loss: 0.3939 - learning_rate: 1.0000e-04
Epoch 31/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.9255 - loss: 0.3665

235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 84ms/step - accuracy: 0.9255 - loss: 0.3665 - val_accuracy: 0.9035 - val_loss: 0.3671 - learning_rate: 1.0000e-04
Epoch 32/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 86ms/step - accuracy: 0.9302 - loss: 0.3445 - val_accuracy: 0.8975 - val_loss: 0.3798 - learning_rate: 1.0000e-04
Epoch 33/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 85ms/step - accuracy: 0.9276 - loss: 0.3531 - val_accuracy: 0.8997 - val_loss: 0.3724 - learning_rate: 1.0000e-04
Epoch 34/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 901ms/step - accuracy: 0.9302 - loss: 0.3543

235/235 ━━━━━━━━━━━━━━━━━━━━ 213s 910ms/step - accuracy: 0.9302 - loss: 0.3544 - val_accuracy: 0.9036 - val_loss: 0.3710 - learning_rate: 1.0000e-04
Epoch 35/100
119/235 ━━━━━━━━━━━━━━━━━━━━ 9s 78ms/step - accuracy: 0.9331 - loss: 0.3292

KeyboardInterrupt: 